# Model Training Pipeline V2
Trains an updated XGBoost regression model using deeper temporal lags and unified stations.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


### 1. Load Preprocessed Data
Load `processed_training_data.csv` (V2 Dataset).


In [ ]:
df = pd.read_csv('processed_training_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
print(f"Data shape: {df.shape}")


### 2. Feature Selection
Define the strict list of features to use. Drop strings, raw datetime, and the target variable itself.


In [ ]:
target_col = 'target_water_level_next'

# V2 FIX: Added alert_level explicitly drop to avoid 'object' crash
string_cols = ['station', 'river_basin', 'rainfall_type', 'status', 'datetime', 'alert_level']
drop_cols = string_cols + [target_col]

features = [col for col in df.columns if col not in drop_cols]
print(f"Number of input features: {len(features)}")


### 3. Time-Based Train-Test Split
Time-series split keeping history intact.


In [ ]:
train_mask = df['datetime'] < '2025-01-01'
test_mask = df['datetime'] >= '2025-01-01'

df_train = df[train_mask]
df_test = df[test_mask]

X_train = df_train[features].copy()
y_train = df_train[target_col]

X_test = df_test[features].copy()
y_test = df_test[target_col]

# Replace inf with nan natively
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print(f"Training sequences: {len(X_train)}")
print(f"Testing sequences: {len(X_test)}")


### 4. Initialize & Train Model (V2 Tuning)
Adjusting hyperparameters to account for the high correlation among massive lag features.


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=200,          # Increased trees to learn more fine interactions
    learning_rate=0.03,        # Dropped LR to prevent rapid overfitting
    max_depth=7,               # Deeper tree to allow more extreme bounds catching
    subsample=0.75,            # Slight drop for variance
    colsample_bytree=0.6,      # Force it to look at random subsets (like 48h rain vs 3h rain)
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model V2 trained successfully!")


### 5. Predict and Evaluate


In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")


### 6. Error Analysis
Look at error distribution across stations.


In [ ]:
df_test = df_test.copy()
df_test['predicted_water_level'] = y_pred
df_test['error_abs'] = np.abs(df_test['target_water_level_next'] - df_test['predicted_water_level'])

station_errors = df_test.groupby('station')['error_abs'].mean().sort_values(ascending=False)
print("Mean Absolute Error by Station:")
print(station_errors)


### 7. Feature Importance


In [ ]:
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'][:20][::-1], feature_importance['importance'][:20][::-1])
plt.title('Top 20 Most Important Features')
plt.xlabel('XGBoost Feature Importance')
plt.show()


### 8. Save Model
Export Version 2 of the model.


In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/waterlevel_xgb_v2.joblib'
joblib.dump(model, model_path)

print(f"V2 Model saved successfully to {model_path}")
